# 2. Basics of Optimization 
## Part B Discrete optimization

Let's have a sandwich structure of length $L$ consisting of $N$ blocks with the same thickness.
Each block can be either air or one of the materials from the library: spf, acoustic foam, melamine foam, felt, glass wool.

Find the best sequence of materials to maximize absorption





In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize

from optimization_utils import *

Define variables needed for the problem:

In [ ]:
N_BLOCKS = 10 # play around with this number to see how it affects the optimization results

block_length = L / N_BLOCKS
lengths_b = [block_length] * N_BLOCKS
# print(f"Block lengths: {lengths_b}")

PALETTE = [ # pick 2-3 materials from the palette, otherwise the optimization will take too long
    MATERIALS["air"],
    # MATERIALS["spf"],
    MATERIALS["acoustic_foam"],
    MATERIALS["melamine_foam"],
    # MATERIALS["felt"],
    # MATERIALS["glass_wool"]
]

N_MATS = len(PALETTE)

Optimization itself:

In [ ]:
# 1. Define discrete problem with integer variable type
class DiscreteSandwichProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=N_BLOCKS,
            n_obj=1,
            xl=0,
            xu=N_MATS - 1,
            vtype=int  # Tells pymoo to handle discrete integer operators
        )

    def _evaluate(self, x, out, *args, **kwargs):
        mat_list = [PALETTE[int(idx)] for idx in x]
        reflection, absorption = compute_spectrum(lengths_b, mat_list)

        # obj = -np.mean(absorption)            # Minimize negative mean absorption
        # obj = np.mean(np.abs(reflection))     # Does minimizing the mean reflection coefficient give the same result?
        obj = -np.mean(absorption[freqs<300]) # Minimize negative mean absorption but only for frequencies below 300 Hz
        # obj = -np.max(absorption)             # Minimize negative max absorption

        out["F"] = obj

# 2. Minimal GA setup
problem = DiscreteSandwichProblem()
algorithm = GA(pop_size=40)

res = minimize(
    problem,
    algorithm,
    termination=('n_gen', 30),
    seed=42,
    verbose=True # enables logging of the optimization process
)

# 3. Extract and display best sequence
best_sequence = res.X.astype(int)
best_materials = [PALETTE[idx] for idx in best_sequence]

In [ ]:
print("Optimal Material Sequence (Front to Back Wall):")
print(" | ".join([m.name for m in best_materials]))

In [ ]:
# Reflection and absorption for optimal case and single-material baselines
r_opt, abs_opt = compute_spectrum(lengths_b, best_materials)
r_spf, abs_spf = compute_spectrum([L], [MATERIALS["spf"]])

# Plot results
plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.plot(freqs, np.abs(r_opt), 'k-', linewidth=2, label='GA Discrete Multi-layer')
plt.plot(freqs, np.abs(r_spf), '--', linewidth=2, label='SPF only')
setup_r_axis()

plt.subplot(122)
plt.plot(freqs, abs_opt, 'k-', linewidth=2, label='GA Discrete Multi-layer')
plt.plot(freqs, abs_spf, '--', linewidth=2, label='SPF only')
setup_abs_axis()
plt.show()

### Sources - Acoustics

If you want more details on the employed Transfer Matrix Method, see
Noé Jiménez, Jean-Philippe Groby, and Vicent Romero-García. “The Transfer Matrix Method in Acoustics”. In: Acoustic Waves in Periodic Structures, Metamaterials, and Porous Media: From Fundamentals to Industrial Applications. Ed. by Noé Jiménez,
Olga Umnova, and Jean-Philippe Groby. Cham: Springer International Publishing, 2021, pp. 103–164. isbn: 978-3-030-84300-7. doi: 10.1007/978-3-030-84300-7_4.

The characterize the porous material, we use effective parameters based on semi-phenomenological model of Johnson–Champoux–Allard–Lafarge (JCAL). 
The material parameters are taken from 
M. Niskanen, J.-P. Groby, A. Duclos, O. Dazel, J. C. Le Roux, N. Poulain, T. Huttunen, T. Lähivaara; Deterministic and statistical characterization of rigid frame porous materials from impedance tube measurements. J. Acoust. Soc. Am. 1 October 2017; 142 (4): 2407–2418. https://doi.org/10.1121/1.5008742